In [1]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
adult = fetch_ucirepo(id=2) 
  
# data (as pandas dataframes) 
X = adult.data.features 
y = adult.data.targets 
  
# metadata 
print(adult.metadata) 
  
# variable information 
print(adult.variables) 


# Combine features and target into one DataFrame
df = X.copy()
df['income'] = y['income']

# Filter for each of given races
df_white = df[df['race'].astype(str).str.strip().eq('White')].reset_index(drop=True)
df_black = df[df['race'].astype(str).str.strip().eq('Black')].reset_index(drop=True)
df_asian = df[df['race'].astype(str).str.strip().eq('Asian-Pac-Islander')].reset_index(drop=True)
df_native = df[df['race'].astype(str).str.strip().eq('Amer-Indian-Eskimo')].reset_index(drop=True)
df_other = df[df['race'].astype(str).str.strip().eq('Other')].reset_index(drop=True)
# Quick check
print(df_white.shape)
print(df_white.head())

{'uci_id': 2, 'name': 'Adult', 'repository_url': 'https://archive.ics.uci.edu/dataset/2/adult', 'data_url': 'https://archive.ics.uci.edu/static/public/2/data.csv', 'abstract': 'Predict whether annual income of an individual exceeds $50K/yr based on census data. Also known as "Census Income" dataset. ', 'area': 'Social Science', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 48842, 'num_features': 14, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Age', 'Income', 'Education Level', 'Other', 'Race', 'Sex'], 'target_col': ['income'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 1996, 'last_updated': 'Tue Sep 24 2024', 'dataset_doi': '10.24432/C5XW20', 'creators': ['Barry Becker', 'Ronny Kohavi'], 'intro_paper': None, 'additional_info': {'summary': "Extraction was done by Barry Becker from the 1994 Census database.  A set of reasonably clean records was extracted using the fol

In [2]:
import numpy as np
import pandas as pd

# changes all of the income into 1 and 0 based on if over 50k or if less than
m_gt = df_white['income'].astype(str).str.contains(r'>\s*50\s*K?', case=False, na=False)
m_le = df_white['income'].astype(str).str.contains(r'<=\s*50\s*K?', case=False, na=False)
df_white['income_binary'] = np.select([m_gt, m_le], [1, 0], default=np.nan)  # or default=0 if you prefer
print(df_white['income_binary'].head())
print(df_white['income_binary'].value_counts(dropna=False))
df_white.head()
df_white.drop(columns=['income'], inplace=True)

0    0.0
1    0.0
2    0.0
3    0.0
4    1.0
Name: income_binary, dtype: float64
income_binary
0.0    31155
1.0    10607
Name: count, dtype: int64


In [4]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

def evaluate_dt_depths(
    df: pd.DataFrame,
    target_col: str = 'income_binary',
    drop_cols: tuple = ('race',),
    depths=range(1, 22),           # try depths 1..21 (adjust as needed)
    test_size: float = 0.2,
    random_state: int = 0,
    dt_kwargs: dict | None = None
) -> pd.DataFrame:
    # --- target ---
    y_raw = df[target_col]
    if y_raw.dtype == 'O':
        s = (y_raw.astype(str).str.strip().str.upper()
             .str.replace(r'\.$', '', regex=True)
             .str.replace(r'\s+', '', regex=True))
        y = s.map({'>50K': 1, '<=50K': 0}).astype(int)
    else:
        y = y_raw.astype(int)

    # --- features ---
    X = df.drop(columns=[target_col, *drop_cols], errors='ignore').copy()

    # split by dtype
    num_cols = X.select_dtypes(include=['number', 'bool']).columns.tolist()
    cat_cols = [c for c in X.columns if c not in num_cols]

    # clean categoricals
    for c in cat_cols:
        X[c] = X[c].astype(str).str.strip()
        X[c] = X[c].replace({'?': np.nan})
        X[c] = X[c].replace(r'(?i)^nan$', np.nan, regex=True)

    # train / test split
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    # education ordinal handling
    edu_order = [[
        "Preschool", "1st-4th", "5th-6th", "7th-8th", "9th", "10th", "11th", "12th",
        "HS-grad", "Some-college", "Assoc-voc", "Assoc-acdm",
        "Bachelors", "Masters", "Prof-school", "Doctorate"
    ]]
    edu_present = ("education" in X.columns)
    if edu_present:
        X["education"] = X["education"].astype("object")
    cat_wo_education = [c for c in cat_cols if c != "education"]

    preproc = ColumnTransformer(
        transformers=[
            ('num', SimpleImputer(strategy='median'), num_cols),
            ('edu', Pipeline([
                ('imp', SimpleImputer(strategy='most_frequent')),
                ('ord', OrdinalEncoder(
                    categories=edu_order,
                    handle_unknown='use_encoded_value',
                    unknown_value=-1
                ))
            ]), ['education'] if edu_present else []),
            ('cat', Pipeline([
                ('imp', SimpleImputer(strategy='most_frequent')),
                ('ohe', OneHotEncoder(handle_unknown='ignore'))
            ]), cat_wo_education),
        ],
        remainder='drop'
    )

    base_dt = dict(
        random_state=random_state,
        class_weight='balanced',   # helps if classes are imbalanced
        splitter='best'            # standard
    )
    if dt_kwargs:
        base_dt.update(dt_kwargs)

    rows = []
    for d in depths:
        clf = Pipeline([
            ('prep', preproc),
            ('dt', DecisionTreeClassifier(max_depth=d, **base_dt))
        ])
        clf.fit(X_tr, y_tr)

        y_pred = clf.predict(X_te)
        acc = accuracy_score(y_te, y_pred)
        prec, rec, f1, _ = precision_recall_fscore_support(
            y_te, y_pred, average=None, labels=[0,1], zero_division=0
        )

        # AUC needs probabilities; DecisionTree supports predict_proba
        try:
            y_proba = clf.predict_proba(X_te)[:, 1]
            auc = roc_auc_score(y_te, y_proba)
        except Exception:
            auc = np.nan

        rows.append({
            'max_depth': d,
            'accuracy': acc,
            'precision_0': prec[0], 'recall_0': rec[0], 'f1_0': f1[0],
            'precision_1': prec[1], 'recall_1': rec[1], 'f1_1': f1[1],
            'roc_auc': auc
        })

    return pd.DataFrame(rows).sort_values('max_depth').reset_index(drop=True)


In [8]:
def dt_feature_importance_on_subset(
    df_subset: pd.DataFrame,
    target_col: str = 'income_binary',
    drop_cols: tuple = ('race',),
    max_depth: int | None = None,
    random_state: int = 0,
    dt_kwargs: dict | None = None,
    aggregate_ohe: bool = True,
    return_model: bool = False
):
    # --- target ---
    y_raw = df_subset[target_col]
    if y_raw.dtype == 'O':
        s = (y_raw.astype(str).str.strip().str.upper()
             .str.replace(r'\.$', '', regex=True)
             .str.replace(r'\s+', '', regex=True))
        y = s.map({'>50K': 1, '<=50K': 0}).astype(int)
    else:
        y = y_raw.astype(int)

    # --- features ---
    X = df_subset.drop(columns=[target_col, *drop_cols], errors='ignore').copy()

    num_cols = X.select_dtypes(include=['number', 'bool']).columns.tolist()
    cat_cols = [c for c in X.columns if c not in num_cols]

    for c in cat_cols:
        X[c] = X[c].astype(str).str.strip()
        X[c] = X[c].replace({'?': np.nan})
        X[c] = X[c].replace(r'(?i)^nan$', np.nan, regex=True)

    edu_order = [[
        "Preschool", "1st-4th", "5th-6th", "7th-8th", "9th", "10th", "11th", "12th",
        "HS-grad", "Some-college", "Assoc-voc", "Assoc-acdm",
        "Bachelors", "Masters", "Prof-school", "Doctorate"
    ]]
    edu_present = ("education" in X.columns)
    if edu_present:
        X["education"] = X["education"].astype("object")
    cat_wo_education = [c for c in cat_cols if c != "education"]

    preproc = ColumnTransformer(
        transformers=[
            ('num', SimpleImputer(strategy='median'), num_cols),
            ('edu', Pipeline([
                ('imp', SimpleImputer(strategy='most_frequent')),
                ('ord', OrdinalEncoder(
                    categories=edu_order,
                    handle_unknown='use_encoded_value',
                    unknown_value=-1
                ))
            ]), ['education'] if edu_present else []),
            ('cat', Pipeline([
                ('imp', SimpleImputer(strategy='most_frequent')),
                ('ohe', OneHotEncoder(handle_unknown='ignore'))
            ]), cat_wo_education),
        ],
        remainder='drop'
    )

    base_dt = dict(
        random_state=random_state,
        class_weight='balanced',
        splitter='best'
    )
    if dt_kwargs:
        base_dt.update(dt_kwargs)
    if max_depth is not None:
        base_dt['max_depth'] = max_depth

    pipe = Pipeline([
        ('prep', preproc),
        ('dt', DecisionTreeClassifier(**base_dt))
    ])
    pipe.fit(X, y)

    importances = pipe.named_steps['dt'].feature_importances_
    prep = pipe.named_steps['prep']

    # If you want raw expanded names (each OHE column separate):
    expanded_names = prep.get_feature_names_out()

    if not aggregate_ohe:
        out_raw = (pd.DataFrame({'feature_expanded': expanded_names,
                                 'importance': importances})
                   .sort_values('importance', ascending=False)
                   .reset_index(drop=True))
        return (pipe, out_raw) if return_model else out_raw

    # Aggregate OHE back to original feature
    num_out_dim = len(num_cols)
    edu_out_dim = 1 if edu_present else 0

    # OHE info
    cat_transformer = prep.named_transformers_['cat']
    if cat_transformer == 'drop' or not cat_wo_education:
        ohe_dims_per_feature = []
        ohe_total_dim = 0
    else:
        ohe = cat_transformer.named_steps['ohe']
        ohe_dims_per_feature = [len(cats) for cats in ohe.categories_]
        ohe_total_dim = sum(ohe_dims_per_feature)

    expected_len = num_out_dim + edu_out_dim + ohe_total_dim
    if len(importances) != expected_len:
        raise RuntimeError(
            f"Shape mismatch: got {len(importances)} importances, expected {expected_len}."
        )

    feat_importance = {}
    idx = 0

    for f in num_cols:
        feat_importance[f] = float(importances[idx])
        idx += 1

    if edu_present:
        feat_importance['education'] = float(importances[idx])
        idx += 1

    for f, width in zip(cat_wo_education, ohe_dims_per_feature):
        feat_importance[f] = float(np.sum(importances[idx:idx+width]))
        idx += width

    feat_importance_df = (pd.DataFrame(
        [{'feature': k, 'importance': v} for k, v in feat_importance.items()]
    ).sort_values('importance', ascending=False).reset_index(drop=True))

    return (pipe, feat_importance_df) if return_model else feat_importance_df


In [ ]:
dt_results = evaluate_dt_depths(df_white) 
print(dt_results)

    max_depth  accuracy  precision_0  recall_0      f1_0  precision_1  \
0           1  0.704298     0.929632  0.653025  0.767157     0.456237   
1           2  0.711720     0.948393  0.648853  0.770536     0.465037   
2           3  0.711720     0.948393  0.648853  0.770536     0.465037   
3           4  0.756375     0.943739  0.716097  0.814308     0.512000   
4           5  0.764875     0.952876  0.720430  0.820508     0.521691   
5           6  0.764755     0.952290  0.720751  0.820499     0.521584   
6           7  0.782114     0.947817  0.749157  0.836859     0.544049   
7           8  0.781276     0.945388  0.750120  0.836510     0.543268   
8           9  0.805579     0.931125  0.798427  0.859686     0.582724   
9          10  0.802825     0.932616  0.792971  0.857143     0.577741   
10         11  0.797917     0.934238  0.784304  0.852731     0.569507   
11         12  0.802586     0.931288  0.793934  0.857143     0.577770   
12         13  0.799832     0.932625  0.788637  0.8

NameError: name 'dt_feature_importance_on_subset' is not defined

In [ ]:

imp_dt = dt_feature_importance_on_subset(
    df_white,
    target_col='income_binary',
    drop_cols=('race',),
    max_depth=20,
    random_state=0,
    aggregate_ohe=True
)
imp_dt.head(20)

NameError: name 'dt_feature_importance_on_subset' is not defined

In [10]:
import numpy as np
import pandas as pd

# changes all of the income into 1 and 0 based on if over 50k or if less than
df_full = df 
full_gt = df_full['income'].astype(str).str.contains(r'>\s*50\s*K?', case=False, na=False)
full_le = df_full['income'].astype(str).str.contains(r'<=\s*50\s*K?', case=False, na=False)
df_full['income_binary'] = np.select([full_gt, full_le], [1, 0], default=np.nan)  # or default=0 if you prefer
print(df_full['income_binary'].head())
print(df_full['income_binary'].value_counts(dropna=False))
df_full.head()
df_full.drop(columns=['income'], inplace=True)

0    0.0
1    0.0
2    0.0
3    0.0
4    0.0
Name: income_binary, dtype: float64
income_binary
0.0    37155
1.0    11687
Name: count, dtype: int64


In [12]:

imp_dt = dt_feature_importance_on_subset(
    df_full,
    target_col='income_binary',
    drop_cols=('income',),
    max_depth=20,
    random_state=0,
    aggregate_ohe=True
)
imp_dt.head(20)

,feature,importance
0,marital-status,0.378094
1,capital-gain,0.120991
2,education,0.107303
3,fnlwgt,0.086808
4,age,0.083638
5,hours-per-week,0.054326
6,occupation,0.046527
7,capital-loss,0.034901
8,workclass,0.023795
9,education-num,0.023787
